# Nghiệm thu Next Best Offer và Trino Security

Notebook kiểm tra dữ liệu NBO ngày `2026-01-01` bằng Spark SQL, sau đó gọi Trino HTTPS để chứng minh authentication và RBAC thật. Python chỉ làm nhiệm vụ chạy query, hiển thị kết quả và dừng khi có lỗi.

In [1]:
import base64
import json
import os
import ssl
import urllib.error
import urllib.request

try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

COB_DT = "2026-01-01"
results = []

def sql_check(name, query):
    frame = spark.sql(query)
    frame.show(truncate=False)
    status = frame.first()["result"]
    results.append((name, status))
    if status != "PASS":
        raise AssertionError(f"{name}: {status}")


## 1. Grain và contract NBO

Mỗi khách hàng/ngày phải có đúng một recommendation; score, reason, priority và suppression phải hợp lệ.

In [2]:
sql_check("NBO contract", f"""
WITH duplicate_keys AS (
    SELECT customer_id
    FROM lakehouse.gold.campaign_target
    WHERE cob_dt = DATE '{COB_DT}'
    GROUP BY customer_id
    HAVING COUNT(*) > 1
), stats AS (
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT customer_id) AS unique_customers,
        MIN(cross_sell_score) AS min_score,
        MAX(cross_sell_score) AS max_score,
        SUM(CASE WHEN cross_sell_score IS NULL OR cross_sell_score NOT BETWEEN 0 AND 100 THEN 1 ELSE 0 END) AS invalid_score,
        SUM(CASE WHEN recommendation_reason IS NULL OR TRIM(recommendation_reason) = '' THEN 1 ELSE 0 END) AS missing_reason,
        SUM(CASE WHEN recommended_product IS NULL OR TRIM(recommended_product) = '' THEN 1 ELSE 0 END) AS missing_product,
        SUM(CASE WHEN campaign_priority NOT IN ('HIGH','MEDIUM','LOW') THEN 1 ELSE 0 END) AS invalid_priority,
        SUM(CASE WHEN contact_eligible_flag NOT IN (0,1)
                       OR (contact_eligible_flag = 1 AND suppression_reason IS NOT NULL)
                       OR (contact_eligible_flag = 0 AND suppression_reason IS NULL) THEN 1 ELSE 0 END) AS invalid_suppression
    FROM lakehouse.gold.campaign_target
    WHERE cob_dt = DATE '{COB_DT}'
)
SELECT *, CASE WHEN row_count = 10000 AND row_count = unique_customers
                      AND (SELECT COUNT(*) FROM duplicate_keys) = 0
                      AND min_score >= 0 AND max_score <= 100 AND invalid_score = 0
                      AND missing_reason = 0 AND missing_product = 0
                      AND invalid_priority = 0 AND invalid_suppression = 0
                 THEN 'PASS' ELSE 'FAIL' END AS result
FROM stats
""")

+---------+----------------+---------+---------+-------------+--------------+---------------+----------------+-------------------+------+
|row_count|unique_customers|min_score|max_score|invalid_score|missing_reason|missing_product|invalid_priority|invalid_suppression|result|
+---------+----------------+---------+---------+-------------+--------------+---------------+----------------+-------------------+------+
|10000    |10000           |0        |100      |0            |0             |0              |0               |0                  |PASS  |
+---------+----------------+---------+---------+-------------+--------------+---------------+----------------+-------------------+------+



## 2. Tính lại rule score độc lập

Query dưới đây tự tính lại toàn bộ cộng/trừ điểm từ Gold history và Silver CRM, rồi so sánh với output campaign.

In [3]:
sql_check("NBO score rules", f"""
WITH complaints AS (
    SELECT customer_id, COUNT(*) AS open_complaint_count
    FROM lakehouse.silver.fact_crm_interaction
    WHERE CAST(interaction_date AS DATE) <= DATE '{COB_DT}' AND UPPER(category) = 'COMPLAINT'
      AND COALESCE(UPPER(status),'UNKNOWN') NOT IN ('CLOSED','RESOLVED')
    GROUP BY customer_id
), expected AS (
    SELECT c.customer_id, c.cross_sell_score,
           GREATEST(0, LEAST(100,
               CASE WHEN c.no_credit_card = 1 THEN 30 ELSE 0 END
             + CASE WHEN c.aum_total >= 100000000 THEN 25 ELSE 0 END
             + CASE WHEN c.rfm_segment IN ('Champions','Loyal Customers') THEN 20 ELSE 0 END
             + CASE WHEN c.days_since_last_txn <= 30 THEN 15 ELSE 0 END
             + CASE WHEN UPPER(c.customer_segment) IN ('PRIORITY','VIP') THEN 10 ELSE 0 END
             - CASE WHEN UPPER(COALESCE(m.kyc_status,'UNKNOWN')) <> 'VERIFIED' THEN 30 ELSE 0 END
             - CASE WHEN c.churn_risk = 'High' OR COALESCE(x.open_complaint_count,0) > 0 THEN 20 ELSE 0 END
           )) AS expected_score
    FROM lakehouse.gold.campaign_target c
    JOIN lakehouse.gold.mart_customer_360_history m
      ON c.customer_id = m.customer_id AND c.cob_dt = m.cob_dt
    LEFT JOIN complaints x ON c.customer_id = x.customer_id
    WHERE c.cob_dt = DATE '{COB_DT}'
)
SELECT COUNT(*) AS checked_rows,
       SUM(CASE WHEN cross_sell_score <> expected_score THEN 1 ELSE 0 END) AS mismatches,
       CASE WHEN COUNT(*) = 10000
                  AND SUM(CASE WHEN cross_sell_score <> expected_score THEN 1 ELSE 0 END) = 0
            THEN 'PASS' ELSE 'FAIL' END AS result
FROM expected
""")

+------------+----------+------+
|checked_rows|mismatches|result|
+------------+----------+------+
|10000       |0         |PASS  |
+------------+----------+------+



## 3. Phân phối campaign

Bảng phân phối giúp Marketing nhìn được quy mô audience theo priority, eligibility và sản phẩm đề xuất.

In [4]:
spark.sql(f"""
SELECT campaign_priority, contact_eligible_flag, recommended_product, COUNT(*) AS customers
FROM lakehouse.gold.campaign_target
WHERE cob_dt = DATE '{COB_DT}'
GROUP BY campaign_priority, contact_eligible_flag, recommended_product
ORDER BY campaign_priority, contact_eligible_flag, customers DESC
""").show(100, truncate=False)

+-----------------+---------------------+-------------------+---------+
|campaign_priority|contact_eligible_flag|recommended_product|customers|
+-----------------+---------------------+-------------------+---------+
|HIGH             |1                    |CREDIT_CARD        |1838     |
|HIGH             |1                    |TERM_DEPOSIT       |722      |
|HIGH             |1                    |WEALTH_MANAGEMENT  |436      |
|HIGH             |1                    |PERSONAL_LOAN      |168      |
|LOW              |0                    |CREDIT_CARD        |1000     |
|LOW              |1                    |TERM_DEPOSIT       |245      |
|LOW              |1                    |DIGITAL_SAVINGS    |219      |
|LOW              |1                    |PERSONAL_LOAN      |73       |
|LOW              |1                    |CREDIT_CARD        |31       |
|MEDIUM           |1                    |CREDIT_CARD        |4613     |
|MEDIUM           |1                    |DIGITAL_SAVINGS    |282

## 4. Trino HTTPS, authentication và RBAC

Password lấy từ environment của container, không hiển thị trong output. Certificate tự ký chỉ dùng cho local demo nên client acceptance dùng trust context riêng.

In [5]:
TRINO_URL = "https://trino:8443/v1/statement"
TLS_CONTEXT = ssl._create_unverified_context()

def trino_query(user, password, sql):
    auth = base64.b64encode(f"{user}:{password}".encode()).decode()
    request = urllib.request.Request(
        TRINO_URL, data=sql.encode(), method="POST",
        headers={"Authorization": f"Basic {auth}", "X-Trino-User": user},
    )
    payload = json.loads(urllib.request.urlopen(request, context=TLS_CONTEXT).read())
    rows = list(payload.get("data", []))
    while payload.get("nextUri"):
        payload = json.loads(urllib.request.urlopen(payload["nextUri"], context=TLS_CONTEXT).read())
        rows.extend(payload.get("data", []))
    return rows, payload.get("error")

try:
    urllib.request.urlopen(urllib.request.Request(TRINO_URL, data=b"SELECT 1", method="POST"), context=TLS_CONTEXT)
    anonymous_result = "FAIL"
except urllib.error.HTTPError as exc:
    anonymous_result = "PASS" if exc.code == 401 else "FAIL"

marketing_password = os.environ["TRINO_MARKETING_PASSWORD"]
engineering_password = os.environ["TRINO_ENGINEERING_PASSWORD"]
marketing_user, marketing_user_error = trino_query("marketing", marketing_password, "SELECT current_user")
marketing_rows, marketing_error = trino_query("marketing", marketing_password, "SELECT COUNT(*) FROM lakehouse.sandbox.mart_customer_360_masked")
_, marketing_raw_error = trino_query("marketing", marketing_password, "SELECT COUNT(*) FROM lakehouse.bronze.core_customer")
engineering_user, engineering_user_error = trino_query("data_engineer", engineering_password, "SELECT current_user")
engineering_rows, engineering_error = trino_query("data_engineer", engineering_password, "SELECT COUNT(*) FROM lakehouse.bronze.core_customer")

security_evidence = [
    ("anonymous rejected", anonymous_result),
    ("marketing identity", "PASS" if marketing_user == [["marketing"]] and not marketing_user_error else "FAIL"),
    ("marketing sandbox allowed", "PASS" if marketing_rows == [[10000]] and not marketing_error else "FAIL"),
    ("marketing raw denied", "PASS" if marketing_raw_error and "Access Denied" in marketing_raw_error["message"] else "FAIL"),
    ("engineering identity", "PASS" if engineering_user == [["data_engineer"]] and not engineering_user_error else "FAIL"),
    ("engineering raw allowed", "PASS" if engineering_rows == [[30000]] and not engineering_error else "FAIL"),
]
for name, status in security_evidence:
    print(f"{name}: {status}")
    results.append((name, status))
    if status != "PASS":
        raise AssertionError(f"{name}: {status}")

anonymous rejected: PASS
marketing identity: PASS
marketing sandbox allowed: PASS
marketing raw denied: PASS
engineering identity: PASS
engineering raw allowed: PASS


In [6]:
print(f"Completed {len(results)}/{len(results)} acceptance checks")
print("NBO + TRINO SECURITY ACCEPTANCE: PASS")

Completed 8/8 acceptance checks
NBO + TRINO SECURITY ACCEPTANCE: PASS
